# Power Consumption Prediction Training

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import seaborn as sns
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import (confusion_matrix, classification_report, 
                           roc_curve, auc, roc_auc_score, accuracy_score, mean_squared_error, mean_absolute_error, r2_score)
from sklearn.utils.class_weight import compute_class_weight
import os
import json
from datetime import datetime, timedelta
import glob
import missingno as msno
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import train_test_split
import xgboost as xgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pickle

## Extract and Combine Data

In [ ]:
def extract_consumption_data(json_data, apartment_id):
    """Extract consumption data from a JSON file"""
    data = {}
    
    data['DateTime'] = json_data.get('Datetime', '')
    data['Hours'] = json_data.get('Hours', '')
    data['User'] = json_data.get('User', '')
    data['ApartmentID'] = apartment_id
    
    # Process Consumptions
    if 'Consumptions' in json_data:
        for location, consumption_data in json_data['Consumptions'].items():
            # Skip excluded devices
            #if location in excluded_devices:
            #    continue
            for key, value in consumption_data.items():
                data[f'{location}_Consumption_{key}'] = value
    
    return data

def process_consumption_data(base_dir, apartment_id, year, selected_days):
    """Process JSON files and extract consumption data for each day without aggregation"""
    print(f"Processing {apartment_id} consumption data...")
    
    # Create paths
    apartment_year_path = os.path.join(base_dir, apartment_id, year)
    print(f"Checking path: {apartment_year_path}")
    
    if not os.path.exists(apartment_year_path):
        print(f"Path not found: {apartment_year_path}")
        return
    
    # Create output directory
    output_dir = f"consumption_data_{apartment_id}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each month and day
    for month in selected_days.keys():
        month_dir = os.path.join(apartment_year_path, month)
        print(f"Checking month directory: {month_dir}")
        
        if not os.path.exists(month_dir):
            print(f"Month directory not found: {month_dir}")
            continue
        
        for day in selected_days[month]:
            day_str = f"{day:02d}"
            day_data = []
            
            # Check if there's a day subfolder
            day_dir = os.path.join(month_dir, day_str)
            if os.path.exists(day_dir):
                # Look in the day folder
                pattern = os.path.join(day_dir, "*.json")
            else:
                # Look directly in the month folder
                pattern = os.path.join(month_dir, f"{day_str}.{month}.{year}*.json")
            
            json_files = glob.glob(pattern)
            print(f"Found {len(json_files)} files for {month}/{day_str}")
            
            # Skip if no files found
            if not json_files:
                continue
            
            # Process all json files for this day
            for json_file in json_files:
                try:
                    with open(json_file, 'r') as f:
                        data = json.load(f)
                    
                    # Extract consumption data
                    consumption_data = extract_consumption_data(data, apartment_id)
                    if consumption_data and any(key for key in consumption_data.keys() if 'Consumption' in key):
                        day_data.append(consumption_data)
                except Exception as e:
                    print(f"Error processing {json_file}: {e}")
            
            # If we have data for this day, process it
            if day_data:
                df = pd.DataFrame(day_data)
                
                # Convert DateTime to datetime objects
                df['DateTime'] = pd.to_datetime(df['DateTime'], format='%d/%m/%Y', errors='coerce')
                
                # Extract hour from Hours column
                if 'Hours' in df.columns:
                    df['Hour'] = df['Hours'].str.split(':').str[0].astype(int)
                else:
                    df['Hour'] = df['DateTime'].dt.hour
                
                # Add time-based features
                df['Date'] = df['DateTime'].dt.strftime('%d/%m/%Y')
                df['DayOfWeek'] = df['DateTime'].dt.dayofweek  # 0 is Monday, 6 is Sunday
                df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)  # 1 for weekends
                
                # Save to CSV - no aggregation, preserving all original data points
                output_filename = os.path.join(output_dir, f"{apartment_id}_{year}_{month}_{day_str}_consumption.csv")
                df.to_csv(output_filename, index=False)
                print(f"Saved raw consumption data to {output_filename}")

def combine_room_data(apartment_id, rooms):
    """Combine daily CSVs into a single file per room"""
    input_dir = f"consumption_data_{apartment_id}"
    
    for room in rooms:
        all_data = []
        
        # Get all CSV files for this apartment
        csv_files = glob.glob(os.path.join(input_dir, f"{apartment_id}_*.csv"))
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                
                # Filter columns for this room
                room_cols = [col for col in df.columns if room in col and 'Consumption' in col]
                
                # Only proceed if we have consumption data for this room
                if room_cols:
                    keep_cols = ['DateTime', 'Date', 'Hour', 'Hours', 'DayOfWeek', 'IsWeekend', 
                                'EntryCounter', 'ApartmentID', 'User'] + room_cols
                    
                    keep_cols = [col for col in keep_cols if col in df.columns]
                    
                    room_df = df[keep_cols].copy()
                    all_data.append(room_df)
            except Exception as e:
                print(f"Error processing {csv_file} for room {room}: {e}")
        
        if all_data:
            # Combine all daily data
            combined_df = pd.concat(all_data, ignore_index=True)
            
            # Save combined data for this room
            output_filename = f"{apartment_id}_{room}_consumption_data.csv"
            combined_df.to_csv(output_filename, index=False)
            print(f"Combined consumption data for {room} saved to {output_filename}")


# Define days to include for each month
selected_days = {
    '05': list(range(1, 32)),
    '06': list(range(1, 32)),
    '07': list(range(1, 32)),
    '08': list(range(1, 32)),
    '09': list(range(1, 32))
}

# Base directory where data is stored
base_dir = "./Data/Sensors"
year = "2023"

# Process apartments
process_consumption_data(base_dir, "Apartment_1", year, selected_days)
process_consumption_data(base_dir, "Apartment_2", year, selected_days)

# Combine room data after processing
combine_room_data("Apartment_1", ["House"])
combine_room_data("Apartment_2", ["House"])

## Display daily consumption

In [ ]:
# ======= Apartment 1 =======
# Load the dataset
df = pd.read_csv('Apartment_1_House_consumption_data.csv')

# Convert DateTime column to datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Extract just the date component (without time)
df['Date'] = df['DateTime'].dt.date

# Group by date and calculate total power consumption per day
daily_power = df.groupby('Date')['House_Consumption_TotalPower'].agg(
    ['mean', 'sum', 'count']
).reset_index()

# Display the results
print(daily_power[['Date', 'sum', 'count']].head(10))

# Create a plot
plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='sum', data=daily_power)
plt.title('Daily Electricity Consumption (sum)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

# Print total consumption statistics
print(f"Total days of data: {len(daily_power)}")
print(f"Average daily consumption: {daily_power['sum'].mean():.2f} Watts")
print(f"Total consumption: {daily_power['sum'].sum():.2f} Watts")

# ======= Apartment 2 =======

# Load the dataset
df = pd.read_csv('Apartment_2_House_consumption_data.csv')

# Convert DateTime column to datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Extract just the date component (without time)
df['Date'] = df['DateTime'].dt.date

# Group by date and calculate total power consumption per day
daily_power = df.groupby('Date')['House_Consumption_TotalPower'].agg(
    ['mean', 'sum', 'count']
).reset_index()

# Display the results
print(daily_power[['Date', 'sum', 'count']].head(10))

# Create a plot
plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='sum', data=daily_power)
plt.title('Daily Electricity Consumption (sum)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True)
plt.show()

# Print total consumption statistics
print(f"Total days of data: {len(daily_power)}")
print(f"Average daily consumption: {daily_power['sum'].mean():.2f} Watts")
print(f"Total consumption: {daily_power['sum'].sum():.2f} Watts")

## Check for duplicates, missing values, outliers

In [ ]:
def check_duplicates(df):
    """Check for duplicates in the dataframe"""
    print("=== DUPLICATE CHECKS ===")
   
    # Check for exact duplicates
    exact_duplicates = df.duplicated().sum()
    print(f"Exact duplicate rows: {exact_duplicates}")
   
    # Check for time-based duplicates if DateTime and Hours columns exist
    if 'DateTime' in df.columns and 'Hours' in df.columns:
        time_duplicates = df.duplicated(subset=['DateTime', 'Hours']).sum()
        print(f"Duplicate DateTime/Hours combinations: {time_duplicates}")
       
        if time_duplicates > 0:
            print("\nFirst few duplicate time entries:")
            return df[df.duplicated(subset=['DateTime', 'Hours'], keep=False)].sort_values(['DateTime', 'Hours']).head()
   
    return None

def check_missing_values(df):
    """Check for missing values in the dataframe"""
    print("\n=== MISSING VALUES CHECKS ===")
   
    # Count missing values per column
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
   
    if len(missing_values) > 0:
        print("Columns with missing values:")
        for col, count in missing_values.items():
            print(f"  {col}: {count} missing values ({count/len(df)*100:.2f}%)")
    else:
        print("No missing values found.")
   
    return missing_values

def check_outliers(df):
    """Check for outliers in the dataframe using IQR method"""
    print("\n=== OUTLIER CHECKS ===")
   
    # Only analyze numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    outlier_info = {}
   
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
       
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
       
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
       
        if outlier_count > 0:
            outlier_pct = outlier_count / len(df) * 100
            outlier_info[col] = {
                'count': outlier_count,
                'percentage': outlier_pct,
                'bounds': (lower_bound, upper_bound)
            }
   
    # Display outlier results
    if outlier_info:
        print(f"Found {len(outlier_info)} columns with outliers.")
        sorted_outliers = sorted(outlier_info.items(), key=lambda x: x[1]['percentage'], reverse=True)
       
        print("\nAll columns with outliers:")
        for col, stats in sorted_outliers:
            print(f"  {col}: {stats['count']} outliers ({stats['percentage']:.2f}%)")
            print(f"    Values outside: [{stats['bounds'][0]:.2f}, {stats['bounds'][1]:.2f}]")
    else:
        print("No outliers found in any column.")
        
    print("\n")
    return outlier_info

# Load data and run analysis
df = pd.read_csv('Apartment_1_House_consumption_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

df = pd.read_csv('Apartment_2_House_consumption_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

## Fix for duplicates, missing values, outliers

In [ ]:
def remove_rows_with_missing_values(df, threshold=None, subset=None):

    # Make a copy of the dataframe
    df_clean = df.copy()
    
    # If subset is specified, check only those columns
    if subset is not None:
        # Ensure all specified columns exist in the dataframe
        subset = [col for col in subset if col in df.columns]
        if not subset:
            print("None of the specified columns exist in the dataframe.")
            return df_clean
    
    # Count original rows
    original_rows = len(df_clean)
    
    if threshold is not None:
        if not 0 <= threshold <= 1:
            raise ValueError("Threshold must be between 0.0 and 1.0")
        
        # Calculate missing values ratio for each row
        if subset:
            missing_ratio = df_clean[subset].isnull().mean(axis=1)
        else:
            missing_ratio = df_clean.isnull().mean(axis=1)
        
        # Remove rows with missing ratio > threshold
        rows_to_drop = missing_ratio[missing_ratio > threshold].index
        if len(rows_to_drop) > 0:
            df_clean = df_clean.drop(index=rows_to_drop)
            rows_removed = len(rows_to_drop)
            print(f"Removed {rows_removed} rows with > {threshold*100}% missing values "
                  f"({rows_removed/original_rows*100:.2f}% of data)")
        else:
            print(f"No rows with > {threshold*100}% missing values found.")
    else:
        # Remove rows with any missing values
        if subset:
            df_clean = df_clean.dropna(subset=subset)
        else:
            df_clean = df_clean.dropna()
        
        # Calculate number of rows removed
        rows_removed = original_rows - len(df_clean)
        if rows_removed > 0:
            print(f"Removed {rows_removed} rows with missing values "
                  f"({rows_removed/original_rows*100:.2f}% of data)")
        else:
            print("No rows with missing values found.")
    
    return df_clean

def remove_duplicates(df):

    # Count duplicates before removal
    dup_count = df.duplicated().sum()
    
    # check for time-based duplicates
    if 'DateTime' in df.columns and 'Hours' in df.columns:
        time_dup_count = df.duplicated(subset=['DateTime', 'Hours']).sum()
        if time_dup_count > 0:
            print(f"Found {time_dup_count} duplicate time entries. Removing...")
            return df.drop_duplicates(subset=['DateTime', 'Hours'], keep='first')
    
    # If no time-based duplicates but we have exact duplicates
    if dup_count > 0:
        print(f"Found {dup_count} exact duplicate rows. Removing...")
        return df.drop_duplicates()
    
    print("No duplicates found.")
    return df

def handle_outliers(df, method='iqr', columns=None, threshold=1.5):

    # Make a copy of the dataframe
    df_clean = df.copy()
    
    # Select columns to check for outliers
    if columns is None:
        columns = df.select_dtypes(include=['int64', 'float64']).columns
        # Exclude columns ending with _Motion
        columns = [col for col in columns if not col.endswith('_Motion')]
    else:
        # Ensure all specified columns exist and are numeric
        columns = [col for col in columns if col in df.columns and
                   pd.api.types.is_numeric_dtype(df[col]) and
                   not col.endswith('_Motion')]
    
    outlier_info = {}
    
    # Process each column
    for col in columns:
        # Skip columns with all identical values
        if df[col].nunique() <= 1:
            continue
        
        # Calculate boundaries for outliers based on selected method
        if method == 'iqr':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            
            # Identify outliers
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_count = len(outliers)
            
            if outlier_count > 0:
                outlier_percentage = outlier_count / len(df) * 100
                outlier_info[col] = {
                    'count': outlier_count,
                    'percentage': outlier_percentage,
                    'bounds': (lower_bound, upper_bound)
                }
                
                # Handle outliers based on specified method
                if method == 'iqr':
                    # Cap values at the boundaries
                    df_clean.loc[df_clean[col] < lower_bound, col] = lower_bound
                    df_clean.loc[df_clean[col] > upper_bound, col] = upper_bound
        
        elif method == 'zscore':
            # Calculate z-scores
            mean = df[col].mean()
            std = df[col].std()
            z_scores = (df[col] - mean) / std
            
            # Identify outliers (|z| > threshold)
            outliers = df[abs(z_scores) > threshold]
            outlier_count = len(outliers)
            
            if outlier_count > 0:
                outlier_percentage = outlier_count / len(df) * 100
                lower_bound = mean - threshold * std
                upper_bound = mean + threshold * std
                
                outlier_info[col] = {
                    'count': outlier_count,
                    'percentage': outlier_percentage,
                    'bounds': (lower_bound, upper_bound)
                }
                
                # Cap values at z-score boundaries
                df_clean.loc[df_clean[col] < lower_bound, col] = lower_bound
                df_clean.loc[df_clean[col] > upper_bound, col] = upper_bound
        
        elif method == 'remove':
            # For this method, we'll collect all outliers first, then remove rows at the end
            if col not in outlier_info:
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                
                lower_bound = Q1 - threshold * IQR
                upper_bound = Q3 + threshold * IQR
                
                # Identify outliers
                outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
                outliers = df[outlier_mask]
                outlier_count = len(outliers)
                
                if outlier_count > 0:
                    outlier_percentage = outlier_count / len(df) * 100
                    outlier_info[col] = {
                        'count': outlier_count,
                        'percentage': outlier_percentage,
                        'bounds': (lower_bound, upper_bound),
                        'mask': outlier_mask
                    }
    
    # If we're removing outlier rows, we need to do it once for all columns
    if method == 'remove' and outlier_info:
        # Combine all outlier masks
        combined_mask = pd.Series(False, index=df.index)
        for col, info in outlier_info.items():
            combined_mask = combined_mask | info['mask']
        
        # Remove rows with outliers
        df_clean = df_clean[~combined_mask]
        rows_removed = sum(combined_mask)
        print(f"Removed {rows_removed} rows containing outliers ({rows_removed/len(df)*100:.2f}% of data)")
    
    # Report on outliers found and handled
    if outlier_info:
        print(f"Found and handled outliers in {len(outlier_info)} columns using {method} method:")
        
        # Sort by percentage to show most affected columns first
        sorted_outliers = sorted(outlier_info.items(), 
                                key=lambda x: x[1]['percentage'] if 'percentage' in x[1] else 0, 
                                reverse=True)
        
        for col, info in sorted_outliers[:10]:  # Show top 10
            if method != 'remove':
                print(f"  - {col}: {info['count']} outliers ({info['percentage']:.2f}%) - capped at [{info['bounds'][0]:.2f}, {info['bounds'][1]:.2f}]")
    else:
        print("No outliers found or handled.")
    
    if len(outlier_info) > 10:
        print(f"  ... and {len(outlier_info) - 10} more columns")
    
    return df_clean, outlier_info

def clean_data(filepath, outlier_method='iqr', outlier_threshold=1.5):

    print(f"Cleaning data from: {filepath}")
    
    # Load the data
    df = pd.read_csv(filepath)
    print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # 1. Remove rows missing values
    df_no_missing = remove_rows_with_missing_values(df)
    print(f"After removing rows with missing values: {df_no_missing.shape[0]} rows, {df_no_missing.shape[1]} columns")
    
    # 2. Remove duplicates
    df_no_duplicates = remove_duplicates(df_no_missing)
    print(f"After removing duplicates: {df_no_duplicates.shape[0]} rows, {df_no_duplicates.shape[1]} columns")
    
    # 3. Handle outliers
    df_clean, outlier_info = handle_outliers(
        df_no_duplicates, 
        method=outlier_method,
        threshold=outlier_threshold
    )
    print(f"Final shape after cleaning: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
    
    return df_clean


df_clean = clean_data('Apartment_1_House_consumption_data.csv')
df_clean.to_csv('Apartment_1_House_consumption_data.csv', index=False)

df_clean = clean_data('Apartment_2_House_consumption_data.csv')
df_clean.to_csv('Apartment_2_House_consumption_data.csv', index=False)


## Removing Columns

In [ ]:
def clean_apartment_data(file_path):

    # Load the dataset
    df = pd.read_csv(file_path)
    
    # List of columns to keep
    columns_to_keep = [
        'DateTime',
        'Hour',
        'DayOfWeek',
        'IsWeekend',
        'House_Consumption_TotalPower' # Main target variable
    ]
    
    # Keep only the useful columns
    df_cleaned = df[columns_to_keep]
    
    # Save the cleaned dataset
    df_cleaned.to_csv(file_path, index=False)


clean_apartment_data('Apartment_1_House_consumption_data.csv')
clean_apartment_data('Apartment_2_House_consumption_data.csv')

## Transform to hourly data

In [ ]:
def aggregate_hourly_data(file_path):

    # Load the cleaned dataset
    df = pd.read_csv(file_path)
    
    # Convert DateTime to proper datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Create a new column for the date without time
    df['Date'] = df['DateTime'].dt.date
    
    # Group by date and hour
    hourly_data = df.groupby(['Date', 'Hour']).agg(
        TotalPower_Sum=('House_Consumption_TotalPower', 'sum'),
        DayOfWeek=('DayOfWeek', 'first'),   # All entries in an hour have the same DayOfWeek
        IsWeekend=('IsWeekend', 'first'),   # All entries in an hour have the same IsWeekend
        EntryCount=('House_Consumption_TotalPower', 'count')  # Count entries per hour
    ).reset_index()
    
    # Format Hours as HH:00
    hourly_data['Hours'] = hourly_data['Hour'].apply(lambda x: f"{x:02d}:00")
    
    # Rename Date column to DateTime to keep the original format
    hourly_data = hourly_data.rename(columns={'Date': 'DateTime'})
    
    # Reorder columns to have DateTime and Hours at the beginning
    columns_order = ['DateTime', 'Hours', 'Hour', 'DayOfWeek', 'IsWeekend',
                    'TotalPower_Sum', 'EntryCount']
    hourly_data = hourly_data[columns_order]
    
    # Check for hours with fewer than expected entries (assuming 1 entry per minute)
    expected_entries = 60  # 60 minutes per hour
    missing_data = hourly_data[hourly_data['EntryCount'] < expected_entries]
    print(f"\nHours with fewer than {expected_entries} entries (potentially missing data):")
    print(missing_data[['DateTime', 'Hours', 'EntryCount']])
    
    # Calculate average entries per hour
    avg_entries = hourly_data['EntryCount'].mean()
    print(f"\nAverage entries per hour: {avg_entries:.2f}")
    
    # Save the hourly aggregated data
    hourly_data.to_csv(file_path, index=False)


aggregate_hourly_data('Apartment_1_House_consumption_data.csv')
aggregate_hourly_data('Apartment_2_House_consumption_data.csv')

## Adding weather data

In [ ]:
def find_closest_weather_file(sensor_date, weather_files):
    """
    Find the closest weather prediction file to a given sensor date
    """
    closest_file = None
    min_diff = float('inf')
    
    sensor_date_only = sensor_date.date() if hasattr(sensor_date, 'date') else sensor_date
    
    for file in weather_files:
        try:
            # Extract date from filename (format: Pred_2023-01-01.csv)
            date_str = os.path.basename(file).split('_')[1].split('.')[0]
            file_date = datetime.strptime(date_str, '%Y-%m-%d').date()
            
            # Calculate day difference
            diff = abs((file_date - sensor_date_only).days)
            
            # Update closest file if this one is closer
            if diff < min_diff:
                min_diff = diff
                closest_file = file
        except Exception as e:
            print(f"Error parsing date from file {file}: {e}")
    
    return closest_file

def find_closest_time_prediction(sensor_date, sensor_time, weather_df, site, measurement, debug=False):
    """
    Find the weather prediction with the closest time for a specific site and measurement
    """
    # Convert sensor_date to string format used in weather file
    sensor_date_str = sensor_date.strftime('%Y-%m-%d')
    
    if debug:
        print(f"\nDEBUG - Looking for: Date={sensor_date_str}, Time={sensor_time}, Site={site}, Measurement={measurement}")
        print(f"Weather file shape: {weather_df.shape}")
        print(f"Weather file columns: {weather_df.columns.tolist()}")
        print(f"Sample of weather data:\n{weather_df.head(3)}")
        print(f"Weather file date format example: {weather_df['Time'].iloc[0] if not weather_df.empty else 'No data'}")
        print(f"Checking if contains {sensor_date_str} in {weather_df['Time'].iloc[0] if not weather_df.empty else 'No data'}")
    
    # Check if 'Time' column exists
    if 'Time' not in weather_df.columns:
        if debug:
            print("No 'Time' column found in weather data")
        return np.nan, ""
        
    # Check if date format matches exactly or needs conversion
    date_format_exact = weather_df['Time'].astype(str).str.contains(sensor_date_str).any()
    date_format_with_dashes = weather_df['Time'].astype(str).str.contains(sensor_date_str.replace('-', '/')).any()
    reversed_date = f"{sensor_date_str.split('-')[2]}-{sensor_date_str.split('-')[1]}-{sensor_date_str.split('-')[0]}"
    date_format_reversed = weather_df['Time'].astype(str).str.contains(reversed_date).any()
    
    if debug:
        print(f"Date format matches - Exact: {date_format_exact}, With slashes: {date_format_with_dashes}, Reversed: {date_format_reversed}")
        print(f"Reversed date format: {reversed_date}")
    
    # Try different date formats
    if date_format_exact:
        date_filtered = weather_df[weather_df['Time'].astype(str).str.contains(sensor_date_str)]
    elif date_format_with_dashes:
        date_filtered = weather_df[weather_df['Time'].astype(str).str.contains(sensor_date_str.replace('-', '/'))]
    elif date_format_reversed:
        date_filtered = weather_df[weather_df['Time'].astype(str).str.contains(reversed_date)]
    else:
        # If no exact match, try with contains
        day = sensor_date_str.split('-')[2]
        month = sensor_date_str.split('-')[1]
        date_filtered = weather_df[weather_df['Time'].astype(str).str.contains(f"{day}-{month}", na=False)]
    
    if debug:
        print(f"After date filter: {len(date_filtered)} rows")
        if not date_filtered.empty:
            print(f"Sample after date filter:\n{date_filtered.head(3)}")
    
    # Check if 'Site' and 'Measurement' columns exist
    if 'Site' not in weather_df.columns or 'Measurement' not in weather_df.columns:
        if debug:
            print("Missing required columns 'Site' or 'Measurement' in weather data")
        return np.nan, ""
    
    site_measurement_filtered = date_filtered[
        (date_filtered['Site'] == site) & 
        (date_filtered['Measurement'] == measurement)
    ]
    
    if debug:
        print(f"After site/measurement filter: {len(site_measurement_filtered)} rows")
        if not site_measurement_filtered.empty:
            print(f"Sample after site/measurement filter:\n{site_measurement_filtered.head(3)}")
    
    if site_measurement_filtered.empty:
        if debug:
            print("No matches found with exact criteria, trying without date filter")
        
        # Try without the exact date match
        site_measurement_filtered = weather_df[
            (weather_df['Site'] == site) & 
            (weather_df['Measurement'] == measurement)
        ]
        
        if site_measurement_filtered.empty:
            if debug:
                print("Still no matches found")
            return np.nan, ""
    
    # Make a copy to avoid warnings
    filtered_df = site_measurement_filtered.copy()
    
    # Check if 'Hour' column exists
    if 'Hour' not in filtered_df.columns:
        if debug:
            print("No 'Hour' column found in weather data")
        return np.nan, ""
    
    # Extract hour and minute from sensor time
    try:
        if isinstance(sensor_time, str) and ':' in sensor_time:
            hour, minute = map(int, sensor_time.split(':')[:2])
        else:
            hour, minute = 0, 0
    except:
        hour, minute = 0, 0
    
    # Calculate minutes since midnight for sensor time
    sensor_minutes = hour * 60 + minute
    
    if debug:
        print(f"Sensor time: {hour}:{minute} ({sensor_minutes} minutes from midnight)")
    
    # Extract hours from 'Hour' column in weather data
    filtered_df['hour_only'] = filtered_df['Hour'].astype(str).apply(
        lambda x: int(x.split(':')[0]) if isinstance(x, str) and ':' in x else 0
    )
    
    # Calculate minutes since midnight for weather times
    filtered_df['minutes_from_midnight'] = filtered_df['hour_only'] * 60
    
    if debug:
        print(f"Hours in weather data: {filtered_df['hour_only'].tolist()}")
        print(f"Minutes from midnight: {filtered_df['minutes_from_midnight'].tolist()}")
    
    # Calculate time difference
    filtered_df['time_diff'] = abs(filtered_df['minutes_from_midnight'] - sensor_minutes)
    
    if debug:
        print(f"Time differences: {filtered_df['time_diff'].tolist()}")
    
    # Find entries with the smallest time difference
    min_time_diff = filtered_df['time_diff'].min()
    closest_time_entries = filtered_df[filtered_df['time_diff'] == min_time_diff].copy()
    
    if debug:
        print(f"Entries with smallest time diff ({min_time_diff} minutes): {len(closest_time_entries)}")
        print(f"These entries:\n{closest_time_entries}")
    
    # Check if 'Value' and 'Unit' columns exist
    if 'Value' not in closest_time_entries.columns:
        if debug:
            print("No 'Value' column found in weather data")
        return np.nan, ""
    
    # If multiple entries with the same time, take the one with highest Prediction value
    if len(closest_time_entries) > 1:
        if 'Prediction' in closest_time_entries.columns:
            closest_time_entries.loc[:, 'Prediction_Numeric'] = pd.to_numeric(
                closest_time_entries['Prediction'], errors='coerce'
            )
            
            if debug:
                print(f"Multiple entries, Prediction values: {closest_time_entries['Prediction'].tolist()}")
                print(f"Numeric Prediction values: {closest_time_entries['Prediction_Numeric'].tolist()}")
            
            # Sort by Prediction value (descending) and take the first row
            closest_entry = closest_time_entries.sort_values('Prediction_Numeric', ascending=False).iloc[0]
        else:
            closest_entry = closest_time_entries.iloc[0]
    else:
        closest_entry = closest_time_entries.iloc[0]
    
    # Get unit (if available)
    unit = closest_entry.get('Unit', '')
    
    if debug:
        print(f"Selected value: {closest_entry['Value']}, unit: {unit}")
    
    return closest_entry['Value'], unit

def add_weather_data(sensor_csv, output_csv, weather_dir, sites):
    """
    Add weather prediction data to apartment sensor readings
    """
    print(f"Processing {sensor_csv}...")
    
    # Check if the sensor file exists
    if not os.path.exists(sensor_csv):
        print(f"Error: Sensor file {sensor_csv} not found")
        return
    
    # Load sensor data
    sensors_df = pd.read_csv(sensor_csv)
    print(f"Loaded sensor data with {len(sensors_df)} rows and {len(sensors_df.columns)} columns")
    
    # Make sure to add this to keep track of whether any weather data was found
    weather_data_found = False
    
    # Store original DateTime format before conversion
    sensors_df['DateTime_Original'] = sensors_df['DateTime']
    
    # Convert DateTime column to datetime - FIX: Use correct format from the file (YYYY-MM-DD)
    print("Converting DateTime column...")
    try:
        sensors_df['DateTime'] = pd.to_datetime(sensors_df['DateTime'], format='%Y-%m-%d', errors='coerce')
    except Exception as e:
        print(f"Error converting DateTime with YYYY-MM-DD format: {e}")
        try:
            # Try alternative format or auto-detection
            sensors_df['DateTime'] = pd.to_datetime(sensors_df['DateTime'], errors='coerce')
        except Exception as e:
            print(f"Error with alternative DateTime format: {e}")
            # Don't use a dummy date, keep original and exit
            print("Could not convert DateTime column, exiting")
            return
    
    # Check if DateTime conversion was successful
    if sensors_df['DateTime'].isna().all():
        print("ERROR: All DateTime values are NaN after conversion")
        # Restore original DateTime values
        sensors_df['DateTime'] = sensors_df['DateTime_Original']
        sensors_df = sensors_df.drop(columns=['DateTime_Original'])
        # Exit the function
        return
    
    # Get list of all weather prediction files
    weather_files = glob.glob(os.path.join(weather_dir, "Pred_*.csv"))
    print(f"Found {len(weather_files)} weather prediction files")
    
    if len(weather_files) == 0:
        print(f"No weather files found in {weather_dir}")
        # Restore original DateTime values and exit
        sensors_df['DateTime'] = sensors_df['DateTime_Original']
        sensors_df = sensors_df.drop(columns=['DateTime_Original'])
        sensors_df.to_csv(output_csv, index=False)
        print(f"Saved original data to {output_csv} (no weather data)")
        return
    
    # Measurements to include with their mapping
    MEASUREMENT_MAPPING = {
        'PRED_GLOB_ctrl': 'global_radiation',
        'PRED_RELHUM_2M_ctrl': 'humidity',
        'PRED_TOT_PREC_ctrl': 'rain',
        'PRED_T_2M_ctrl': 'temperature'
    }
    
    measurements = list(MEASUREMENT_MAPPING.keys())
    
    # Add columns for each site and measurement with empty values
    for site in sites:
        for measurement in measurements:
            # Use the mapped name instead of the original
            mapped_name = MEASUREMENT_MAPPING[measurement]
            col_name = f"{site}-{mapped_name}"  # We'll add the unit later
            sensors_df[col_name] = np.nan
    
    # Track which weather files we've already loaded
    weather_data_cache = {}
    
    # Process each row
    print("Processing rows...")
    
    for idx, row in sensors_df.iterrows():
        sensor_date = row['DateTime']
        sensor_time = row.get('Hours', '00:00')  # Use get() to handle missing column
        
        # Skip if date is invalid
        if pd.isna(sensor_date):
            continue
        
        # Enable debugging for first few rows
        debug_this_row = idx < 5
        
        if debug_this_row:
            print(f"\n==== DEBUG FOR ROW {idx} ====")
            print(f"DateTime: {sensor_date}, Hours: {sensor_time}")
        
        try:
            # Find the closest weather file for this date
            closest_file = find_closest_weather_file(sensor_date, weather_files)
            
            if debug_this_row:
                print(f"Closest weather file: {closest_file}")
            
            if closest_file is None:
                print(f"No weather file found for date {sensor_date}")
                continue
                
            # Load the weather data (with caching to avoid reloading)
            if closest_file in weather_data_cache:
                weather_df = weather_data_cache[closest_file]
            else:
                if debug_this_row:
                    print(f"Loading weather file: {closest_file}")
                    
                try:
                    weather_df = pd.read_csv(closest_file)
                    
                    # Rename columns if necessary
                    if 'Site' not in weather_df.columns and len(weather_df.columns) >= 6:
                        column_names = ['Time', 'Value', 'Prediction', 'Site', 'Measurement', 'Unit', 'Hour']
                        weather_df.columns = column_names[:len(weather_df.columns)]
                    
                    # Print some details about the file contents
                    if debug_this_row:
                        print(f"Weather file shape: {weather_df.shape}")
                        print(f"Weather file columns: {weather_df.columns.tolist()}")
                except Exception as e:
                    print(f"Error loading weather file {closest_file}: {e}")
                    continue
                
                weather_data_cache[closest_file] = weather_df
            
            # For each site and measurement, find closest prediction
            for site in sites:
                for measurement in measurements:
                    try:
                        value, unit = find_closest_time_prediction(
                            sensor_date, sensor_time, weather_df, site, measurement, 
                            debug=debug_this_row
                        )
                        
                        if not pd.isna(value):
                            # We found data! Set the flag
                            weather_data_found = True
                        
                        # Use the mapped name instead of the original
                        mapped_name = MEASUREMENT_MAPPING[measurement]
                        
                        # Update the column name with unit
                        col_name = f"{site}-{mapped_name}"
                        if unit:  # Add unit to column name if available
                            col_name_with_unit = f"{site}-{mapped_name}-{unit}"
                            # Create column if it doesn't exist
                            if col_name_with_unit not in sensors_df.columns:
                                sensors_df[col_name_with_unit] = np.nan
                            sensors_df.loc[idx, col_name_with_unit] = value
                        else:
                            # Use the original column without unit
                            sensors_df.loc[idx, col_name] = value
                    except Exception as e:
                        if debug_this_row:
                            print(f"Error processing measurement {measurement} for site {site} at row {idx}: {str(e)}")
                    
        except Exception as e:
            if debug_this_row:
                print(f"Error processing row {idx}: {str(e)}")
        
        # Print progress
        if idx % 1000 == 0 and idx > 0:
            print(f"Processed {idx} of {len(sensors_df)} rows...")
    
    # Only remove empty columns if we found weather data
    if weather_data_found:
        # Remove columns that were never filled (all NaN)
        for col in list(sensors_df.columns):
            if col.startswith(tuple(f"{site}-" for site in sites)) and sensors_df[col].isna().all():
                print(f"Removing unused column: {col}")
                sensors_df = sensors_df.drop(columns=[col])
    else:
        print("WARNING: No weather data was found or added to the dataset")
        
    # Restore original DateTime column
    sensors_df['DateTime'] = sensors_df['DateTime_Original']
    sensors_df = sensors_df.drop(columns=['DateTime_Original'])
    
    # Save the result
    sensors_df.to_csv(output_csv, index=False)
    print(f"Saved results to {output_csv}")


weather_dir = "./Data/Weather"

sensor_csv_1 = "Apartment_1_House_consumption_data.csv"
output_csv_1 = "Apartment_1_House_consumption_data.csv"
add_weather_data(sensor_csv_1, output_csv_1, weather_dir, sites=['Sion'])
sensor_csv_2 = "Apartment_2_House_consumption_data.csv"
output_csv_2 = "Apartment_2_House_consumption_data.csv"
add_weather_data(sensor_csv_2, output_csv_2, weather_dir, sites=['Sion'])

## Visualizations

In [ ]:
# Load the data
df = pd.read_csv('Apartment_1_House_consumption_data.csv')
#df = pd.read_csv('Apartment_2_House_consumption_data.csv')

# Convert DateTime and Hours to datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Create a proper timestamp combining date and hour
df['Timestamp'] = pd.to_datetime(df['DateTime'].astype(str) + ' ' + df['Hours'])

# Set better column names for plotting
column_mapping = {
    'TotalPower_Sum': 'Energy Consumption (Watts)',
    'Sion-global_radiation-Watt/m2': 'Global Radiation (W/m²)',
    'Sion-humidity-Percent': 'Humidity (%)',
    'Sion-rain-Kg/m2': 'Rainfall (kg/m²)',
    'Sion-temperature-°C': 'Temperature (°C)'
}

# Create shorter names for the variables
df = df.rename(columns=column_mapping)

# ============================================================================== 1. Hourly pattern visualization (average by hour of day) ========================================================================
plt.figure(figsize=(15, 12))

# Energy consumption by hour of day
plt.subplot(3, 2, 1)
hourly_avg = df.groupby('Hour')['Energy Consumption (Watts)'].mean()
sns.barplot(x=hourly_avg.index, y=hourly_avg.values)
plt.title('Average Energy Consumption by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Energy Consumption (Watts)')
plt.xticks(range(0, 24, 2))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Temperature by hour of day
plt.subplot(3, 2, 2)
hourly_temp = df.groupby('Hour')['Temperature (°C)'].mean()
sns.barplot(x=hourly_temp.index, y=hourly_temp.values)
plt.title('Average Temperature by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Temperature (°C)')
plt.xticks(range(0, 24, 2))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Humidity by hour of day
plt.subplot(3, 2, 3)
hourly_humidity = df.groupby('Hour')['Humidity (%)'].mean()
sns.barplot(x=hourly_humidity.index, y=hourly_humidity.values)
plt.title('Average Humidity by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Humidity (%)')
plt.xticks(range(0, 24, 2))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Global radiation by hour of day
plt.subplot(3, 2, 4)
hourly_radiation = df.groupby('Hour')['Global Radiation (W/m²)'].mean()
sns.barplot(x=hourly_radiation.index, y=hourly_radiation.values)
plt.title('Average Global Radiation by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Global Radiation (W/m²)')
plt.xticks(range(0, 24, 2))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Rainfall by hour of day
plt.subplot(3, 2, 5)
hourly_rain = df.groupby('Hour')['Rainfall (kg/m²)'].mean()
sns.barplot(x=hourly_rain.index, y=hourly_rain.values)
plt.title('Average Rainfall by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Rainfall (kg/m²)')
plt.xticks(range(0, 24, 2))
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# ============================================================================== 2. Daily time series visualization ========================================================================
# Aggregate data by day
daily_df = df.groupby(df['DateTime'].dt.date).agg({
    'Energy Consumption (Watts)': 'sum',
    'Temperature (°C)': 'mean',
    'Humidity (%)': 'mean',
    'Global Radiation (W/m²)': 'mean',
    'Rainfall (kg/m²)': 'sum'
}).reset_index()
daily_df['DateTime'] = pd.to_datetime(daily_df['DateTime'])

plt.figure(figsize=(15, 15))

# Daily energy consumption
plt.subplot(5, 1, 1)
plt.plot(daily_df['DateTime'], daily_df['Energy Consumption (Watts)'], marker='o', linestyle='-')
plt.title('Daily Energy Consumption')
plt.ylabel('Energy Consumption (Watts)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Daily temperature
plt.subplot(5, 1, 2)
plt.plot(daily_df['DateTime'], daily_df['Temperature (°C)'], marker='o', linestyle='-', color='red')
plt.title('Daily Average Temperature')
plt.ylabel('Temperature (°C)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Daily humidity
plt.subplot(5, 1, 3)
plt.plot(daily_df['DateTime'], daily_df['Humidity (%)'], marker='o', linestyle='-', color='blue')
plt.title('Daily Average Humidity')
plt.ylabel('Humidity (%)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Daily global radiation
plt.subplot(5, 1, 4)
plt.plot(daily_df['DateTime'], daily_df['Global Radiation (W/m²)'], marker='o', linestyle='-', color='orange')
plt.title('Daily Average Global Radiation')
plt.ylabel('Global Radiation (W/m²)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Daily rainfall
plt.subplot(5, 1, 5)
plt.bar(daily_df['DateTime'], daily_df['Rainfall (kg/m²)'], color='skyblue')
plt.title('Daily Total Rainfall')
plt.ylabel('Rainfall (kg/m²)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# ============================================================================== 3. Time series visualization for a week (to see patterns more clearly) ========================================================================
# Select a week of data for detailed visualization
# First get the first complete week in the dataset
start_date = df['DateTime'].min()
# Find the next Monday
while start_date.dayofweek != 0:  # 0 is Monday
    start_date += pd.Timedelta(days=1)
    
end_date = start_date + pd.Timedelta(days=6)
week_data = df[(df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)]

plt.figure(figsize=(15, 12))

# Energy consumption for one week
plt.subplot(5, 1, 1)
plt.plot(week_data['Timestamp'], week_data['Energy Consumption (Watts)'], marker='.', linestyle='-')
plt.title('Energy Consumption (One Week)')
plt.ylabel('Watts')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Temperature for one week
plt.subplot(5, 1, 2)
plt.plot(week_data['Timestamp'], week_data['Temperature (°C)'], marker='.', linestyle='-', color='red')
plt.title('Temperature (One Week)')
plt.ylabel('°C')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Humidity for one week
plt.subplot(5, 1, 3)
plt.plot(week_data['Timestamp'], week_data['Humidity (%)'], marker='.', linestyle='-', color='blue')
plt.title('Humidity (One Week)')
plt.ylabel('%')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Global radiation for one week
plt.subplot(5, 1, 4)
plt.plot(week_data['Timestamp'], week_data['Global Radiation (W/m²)'], marker='.', linestyle='-', color='orange')
plt.title('Global Radiation (One Week)')
plt.ylabel('W/m²')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

# Rainfall for one week
plt.subplot(5, 1, 5)
plt.bar(week_data['Timestamp'], week_data['Rainfall (kg/m²)'], color='skyblue')
plt.title('Rainfall (One Week)')
plt.ylabel('kg/m²')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# ============================================================================== 4. Correlation visualization ========================================================================
correlation_df = df[['Energy Consumption (Watts)', 'Temperature (°C)', 
                    'Humidity (%)', 'Global Radiation (W/m²)', 
                    'Rainfall (kg/m²)']]
correlation_matrix = correlation_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Between Energy Consumption and Weather Variables')
plt.tight_layout()
plt.show()


## Test stationarity

In [ ]:
# Load the data
df = pd.read_csv('Apartment_1_House_consumption_data.csv')
#df = pd.read_csv('Apartment_2_House_consumption_data.csv')

# Convert DateTime to datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Set datetime as index
df_indexed = df.set_index('DateTime')

# Function to test stationarity
def test_stationarity(series, title, significance=0.05):
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot the time series
    ax1.plot(series, label='Original')
    ax1.set_title(f'Time Series Analysis: {title}')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Value')
    
    # Calculate and plot rolling statistics
    rolling_mean = series.rolling(window=24).mean()  # 24-hour window
    rolling_std = series.rolling(window=24).std()
    
    ax1.plot(rolling_mean, label='Rolling Mean (24h)', color='red')
    ax1.plot(rolling_std, label='Rolling Std (24h)', color='green')
    ax1.legend(loc='best')
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Perform Augmented Dickey-Fuller test
    result = adfuller(series.dropna())
    
    # Extract and format ADF test results
    adf_stat = result[0]
    p_value = result[1]
    critical_values = result[4]
    
    # Create text for ADF test results
    adf_text = f'ADF Statistic: {adf_stat:.4f}\n'
    adf_text += f'p-value: {p_value:.4f}\n'
    adf_text += 'Critical Values:\n'
    for key, value in critical_values.items():
        adf_text += f'   {key}: {value:.4f}\n'
    
    # Highlight stationarity result
    is_stationary = p_value < significance
    result_text = f'Result: The series is {"stationary" if is_stationary else "non-stationary"}'
    adf_text += f'\n{result_text}'
    
    # Plot autocorrelation and partial autocorrelation
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    
    plot_acf(series.dropna(), ax=ax2, lags=48)  # Show 48 lags (2 days for hourly data)
    ax2.set_title('Autocorrelation Function')
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    # Add ADF test results as text
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    fig.text(0.15, 0.15, adf_text, fontsize=12, bbox=props)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for text
    plt.show()
    
    return is_stationary, p_value, adf_stat

# Test stationarity of TotalPower_Sum consumption
is_stationary, p_value, adf_stat = test_stationarity(df_indexed['TotalPower_Sum'], 'Power Consumption (Watts)')

# If you want to test other variables as well
test_stationarity(df_indexed['Sion-temperature-°C'], 'Temperature (°C)')

# Test seasonal differences (if it's non-stationary)
if not is_stationary:
    print("\nTesting first difference...")
    diff1 = df_indexed['TotalPower_Sum'].diff().dropna()
    test_stationarity(diff1, 'First Difference of Power Consumption')
    
    print("\nTesting seasonal difference (24-hour)...")
    seasonal_diff = df_indexed['TotalPower_Sum'].diff(24).dropna()
    test_stationarity(seasonal_diff, 'Seasonal Difference (24h) of Power Consumption')
    
    print("\nTesting both first and seasonal difference...")
    diff_both = df_indexed['TotalPower_Sum'].diff().diff(24).dropna()
    test_stationarity(diff_both, 'First and Seasonal Difference of Power Consumption')

# Additional visualization: Decompose the time series
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose the series
try:
    decomposition = seasonal_decompose(df_indexed['TotalPower_Sum'], model='additive', period=24)  # 24 for hourly data
    
    plt.figure(figsize=(14, 12))
    plt.subplot(411)
    plt.plot(decomposition.observed)
    plt.title('Original Series')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.subplot(412)
    plt.plot(decomposition.trend)
    plt.title('Trend Component')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.subplot(413)
    plt.plot(decomposition.seasonal)
    plt.title('Seasonal Component (24h)')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.subplot(414)
    plt.plot(decomposition.resid)
    plt.title('Residual Component')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Test if residuals are stationary
    print("\nTesting stationarity of the residual component...")
    test_stationarity(decomposition.resid.dropna(), 'Residual Component')
    
except Exception as e:
    print(f"Could not perform seasonal decomposition: {e}")
    print("This might be due to missing values or insufficient data.")

## Feature Engineering: Seasons, Lag, Lead, Rolling features

In [ ]:
def engineer_apartment_features(file_path):

    # Load the dataset
    df = pd.read_csv(file_path)

    # Convert DateTime to proper datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])

    # Create a proper timestamp combining date and hour
    df['Timestamp'] = pd.to_datetime(df['DateTime'].astype(str) + ' ' + df['Hours'])

    # Sort data by timestamp to ensure proper time ordering
    df = df.sort_values('Timestamp')

    # Display information about the dataset
    print(f"Dataset has {len(df)} rows and spans from {df['DateTime'].min()} to {df['DateTime'].max()}")
    print(f"Number of unique dates: {df['DateTime'].nunique()}")

    # Seasonal Features
    # Add a simple season feature based on month
    seasons = {
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Fall', 10: 'Fall', 11: 'Fall'
    }
    df['Month'] = df['DateTime'].dt.month
    df['Season'] = df['Month'].map(seasons)

    # Create dummy variables for Season (one-hot encoding)
    season_dummies = pd.get_dummies(df['Season'], prefix='Season')

    # Make sure all seasons are represented, even if not in data
    for season in ['Winter', 'Spring', 'Summer', 'Fall']:
        if f'Season_{season}' not in season_dummies.columns:
            season_dummies[f'Season_{season}'] = 0
        else:
            # Ensure all values are stored as integers (0 or 1)
            season_dummies[f'Season_{season}'] = season_dummies[f'Season_{season}'].astype(int)

    # Add season dummies to dataframe
    df = pd.concat([df, season_dummies], axis=1)

    # Remove the Season column
    df = df.drop('Season', axis=1)

    # Lag and Lead Features
    # Sort by timestamp
    df_sorted = df.sort_values('Timestamp')

    # Create lag features (1-hour and 3-hour)
    df_sorted['TotalPower_Sum_lag_1h'] = df_sorted['TotalPower_Sum'].shift(1)
    df_sorted['TotalPower_Sum_lag_3h'] = df_sorted['TotalPower_Sum'].shift(3)

    # Create lead features (1-hour and 3-hour ahead)
    df_sorted['TotalPower_Sum_lead_1h'] = df_sorted['TotalPower_Sum'].shift(-1)
    df_sorted['TotalPower_Sum_lead_3h'] = df_sorted['TotalPower_Sum'].shift(-3)

    # Check for time continuity to handle missing hours
    time_diff_1h = (df_sorted['Timestamp'] - df_sorted['Timestamp'].shift(1)).dt.total_seconds() / 3600
    mask_invalid_1h = (time_diff_1h > 1.5)  # If more than 1.5 hours difference, the lag isn't valid
    df_sorted.loc[mask_invalid_1h, 'TotalPower_Sum_lag_1h'] = np.nan

    # For 3-hour lag, check the time difference
    time_diff_3h = (df_sorted['Timestamp'] - df_sorted['Timestamp'].shift(3)).dt.total_seconds() / 3600
    mask_invalid_3h = (abs(time_diff_3h - 3) > 1.5)  # If not close to 3 hours, the lag isn't valid
    df_sorted.loc[mask_invalid_3h, 'TotalPower_Sum_lag_3h'] = np.nan

    # For 1-hour lead, check the time difference
    lead_time_diff_1h = (df_sorted['Timestamp'].shift(-1) - df_sorted['Timestamp']).dt.total_seconds() / 3600
    mask_invalid_lead_1h = (lead_time_diff_1h > 1.5)  # If more than 1.5 hours difference, the lead isn't valid
    df_sorted.loc[mask_invalid_lead_1h, 'TotalPower_Sum_lead_1h'] = np.nan

    # For 3-hour lead, check the time difference
    lead_time_diff_3h = (df_sorted['Timestamp'].shift(-3) - df_sorted['Timestamp']).dt.total_seconds() / 3600
    mask_invalid_lead_3h = (abs(lead_time_diff_3h - 3) > 1.5)  # If not close to 3 hours, the lead isn't valid
    df_sorted.loc[mask_invalid_lead_3h, 'TotalPower_Sum_lead_3h'] = np.nan

    # Update the original dataframe with the lag and lead features
    df = df_sorted.copy()

    # Rolling Features
    # Create rolling mean features with 3-hour and 6-hour windows
    df['TotalPower_Sum_roll_3h_mean'] = df['TotalPower_Sum'].rolling(window=3, min_periods=1).mean()
    df['TotalPower_Sum_roll_6h_mean'] = df['TotalPower_Sum'].rolling(window=6, min_periods=1).mean()

    # Replace NaN with appropriate values
    # For lag and lead features
    lag_lead_cols = ['TotalPower_Sum_lag_1h', 'TotalPower_Sum_lag_3h', 
                     'TotalPower_Sum_lead_1h', 'TotalPower_Sum_lead_3h']
    for col in lag_lead_cols:
        # Fill with column mean
        df[col] = df[col].fillna(df[col].mean())

    # For rolling features
    roll_cols = ['TotalPower_Sum_roll_3h_mean', 'TotalPower_Sum_roll_6h_mean']
    for col in roll_cols:
        # Fill with column mean
        df[col] = df[col].fillna(df[col].mean())

    # Save the enhanced dataset
    df.to_csv(file_path, index=False)

    return df

engineer_apartment_features('Apartment_1_House_consumption_data.csv')
engineer_apartment_features('Apartment_2_House_consumption_data.csv')

## Define Features and Target

In [ ]:
# Load the feature-engineered dataset
df = pd.read_csv('Apartment_1_House_consumption_data.csv')
#df = pd.read_csv('Apartment_2_House_consumption_data.csv')

# Convert DateTime to proper datetime format for feature extraction
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Extract additional time components
df['Year'] = df['DateTime'].dt.year
df['Day'] = df['DateTime'].dt.day
df['DayOfYear'] = df['DateTime'].dt.dayofyear

# Define the target variable
target = 'TotalPower_Sum'

# Define the features to use in the model
# Exclude non-feature columns and the target variable
non_feature_cols = [
    'DateTime',       # Date string
    'Hours',          # Time string
    'Timestamp',      # Combined date and time
    'kWh',            # Alternative representation of the target
    'TotalPower_Sum', # Target variable
    'EntryCount'      # Count of entries
]

# Define our feature groups
time_features = ['Hour', 'DayOfWeek', 'IsWeekend', 'Month', 'Year', 'Day', 'DayOfYear']
seasonal_features = ['Season_Fall', 'Season_Spring', 'Season_Summer', 'Season_Winter']
weather_features = [
    'Sion-global_radiation-Watt/m2',
    'Sion-humidity-Percent',
    'Sion-rain-Kg/m2',
    'Sion-temperature-°C'
]
lag_features = ['TotalPower_Sum_lag_1h', 'TotalPower_Sum_lag_3h']
lead_features = ['TotalPower_Sum_lead_1h', 'TotalPower_Sum_lead_3h']
rolling_features = ['TotalPower_Sum_roll_3h_mean', 'TotalPower_Sum_roll_6h_mean']

# Combining all feature groups
feature_groups = {
    'Time': time_features,
    'Seasonal': seasonal_features,
    'Weather': weather_features,
    'Lag': lag_features,
    'Lead': lead_features,
    'Rolling': rolling_features
}

# Create a flat list of all features
features = []
for group in feature_groups.values():
    features.extend(group)

## Train and Test split

In [ ]:
# Make sure we have datetime format
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Sort the dataframe by DateTime to ensure chronological order
df = df.sort_values('DateTime')

# Display date range of the dataset
print(f"Dataset spans from {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"Total number of data points: {len(df)}")

# Calculate the split point - using the last 20% of the data for testing
split_percentage = 0.2
split_index = int(len(df) * (1 - split_percentage))

# Split the data chronologically
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Split into features and target
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

## Train-Test: Random forest

In [ ]:
# Initialize the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=150,            # Increased number of trees for more stability
    criterion='squared_error',   # Standard criterion for regression
    max_depth=15,                # Limit depth to prevent overfitting
    min_samples_split=5,         # Require more samples to split a node
    min_samples_leaf=4,          # Require more samples in leaf nodes
    max_features=0.7,            # Use 70% of features at each split
    bootstrap=True,              # Use bootstrap samples
    oob_score=True,              # Calculate out-of-bag score for better validation
    n_jobs=-1,                   # Use all cores
    random_state=42              # For reproducibility
)

# Train the model
print("Training the model...")
rf_model.fit(X_train, y_train)

# Make predictions on training and test sets
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Evaluate the model
print("\nModel Evaluation:")
print("-" * 50)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, set_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Calculate MAPE (Mean Absolute Percentage Error)
    # Avoid division by zero by adding a small epsilon where y_true is zero
    epsilon = 1e-10
    abs_percentage_errors = np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon)) * 100
    mape = np.mean(abs_percentage_errors)
    
    print(f"{set_name} Set Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    print(f"R² Score: {r2:.4f}")
    print("-" * 30)
    
    return mae, rmse, r2, mape

# Calculate metrics for both sets
train_mae, train_rmse, train_r2, train_mape = calculate_metrics(y_train, y_train_pred, "Training")
test_mae, test_rmse, test_r2, test_mape = calculate_metrics(y_test, y_test_pred, "Test")

# Feature importance
print("\nFeature Importance:")
print("-" * 50)

# Get feature importances
feature_importances = rf_model.feature_importances_
features_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

# Display top 15 most important features
print(features_df.head(15))

In [ ]:
os.makedirs('models', exist_ok=True)
model_filename = "models/random_forest_power_consumption_app1.pkl"
#model_filename = "models/random_forest_power_consumption_app2.pkl"

with open(model_filename, 'wb') as f:
    pickle.dump(rf_model, f)

## Train-Test: XGBoost

In [ ]:
# Create DMatrix objects for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set parameters with stronger regularization
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.01,          # Slower learning rate
    'max_depth': 6,                 # Reduced tree depth
    'min_child_weight': 5,          # Require more data in leaf nodes
    'subsample': 0.8,               # Use 80% of data per tree
    'colsample_bytree': 0.7,        # Use 70% of features per tree
    'gamma': 0.2,                   # Increased min loss reduction for split
    'alpha': 0.5,                   # Increased L1 regularization
    'lambda': 2.0,                  # Increased L2 regularization
    'seed': 42
}

eval_list = [(dtrain, 'train'), (dtest, 'test')]
num_rounds = 400  # Increase max rounds (we lowered the learning rate)

bst = xgb.train(
    params, 
    dtrain, 
    num_rounds, 
    evals=eval_list,
    early_stopping_rounds=30,  # Increased early stopping patience
    verbose_eval=False
)

# Store the trained booster directly
best_iteration = bst.best_iteration
print(f"Best iteration: {best_iteration}")

# Make predictions on training and test sets
y_train_pred = bst.predict(dtrain)
y_test_pred = bst.predict(dtest)

# Evaluate the model
print("\nModel Evaluation:")
print("-" * 50)

# Function to calculate metrics
def calculate_metrics(y_true, y_pred, set_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Calculate MAPE (Mean Absolute Percentage Error)
    # Avoid division by zero by adding a small epsilon where y_true is zero
    epsilon = 1e-10
    abs_percentage_errors = np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon)) * 100
    mape = np.mean(abs_percentage_errors)
    
    print(f"{set_name} Set Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    print(f"R² Score: {r2:.4f}")
    print("-" * 30)
    
    return mae, rmse, r2, mape

# Calculate metrics for both sets
train_mae, train_rmse, train_r2, train_mape = calculate_metrics(y_train, y_train_pred, "Training")
test_mae, test_rmse, test_r2, test_mape = calculate_metrics(y_test, y_test_pred, "Test")

# Feature importance
print("\nFeature Importance:")
print("-" * 50)

# Get feature importances
importance_type = 'gain'  # 'gain' for the contribution to the model, 'weight' for number of splits
feature_importances = bst.get_score(importance_type=importance_type)

# Convert to dataframe for easier handling
importance_df = pd.DataFrame({
    'Feature': list(feature_importances.keys()),
    'Importance': list(feature_importances.values())
}).sort_values('Importance', ascending=False)

# If not all features appear (XGBoost only includes used features),
# add missing features with zero importance
missing_features = set(features) - set(importance_df['Feature'])
for feature in missing_features:
    importance_df = pd.concat([importance_df, pd.DataFrame({'Feature': [feature], 'Importance': [0]})], ignore_index=True)

# Sort again after adding missing features
importance_df = importance_df.sort_values('Importance', ascending=False)

# Display top 15 most important features
print("Top 15 features by importance:")
print(importance_df.head(15))

# Calculate and plot feature importance
plt.subplot(1, 2, 2)
top_features = importance_df.head(10)
plt.barh(np.arange(len(top_features)), top_features['Importance'], align='center')
plt.yticks(np.arange(len(top_features)), top_features['Feature'])
plt.xlabel('Importance')
plt.title('Top 10 Important Features')
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('models', exist_ok=True)
model_filename = "models/xgboost_power_consumption_app1.pkl"
#model_filename = "models/xgboost_power_consumption_app2.pkl"

with open(model_filename, 'wb') as f:
    pickle.dump(bst, f)

## Train-Test: Sarimax

In [ ]:
# Ensure data is sorted by datetime
train_df = train_df.sort_values('DateTime')
test_df = test_df.sort_values('DateTime')

print(f"Training data: {len(train_df)} samples from {train_df['DateTime'].min()} to {train_df['DateTime'].max()}")
print(f"Test data: {len(test_df)} samples from {test_df['DateTime'].min()} to {test_df['DateTime'].max()}")

# Select a subset of exogenous features
exog_features = [
    'Hour', 'DayOfWeek', 'IsWeekend',  # Time features
    'Season_Summer', 'Season_Winter', 'Season_Fall', 'Season_Spring',  # Season features
    'Sion-temperature-°C',             # Temperature is often highly correlated with energy use
    'Sion-global_radiation-Watt/m2',    # Solar radiation affects energy use
    'Sion-humidity-Percent',
    'Sion-rain-Kg/m2'
]

# Print selected features
print(f"\nSelected exogenous features for SARIMAX: {exog_features}")

# Extract target and exogenous variables
y_train_sarimax = train_df[target].values
X_train_sarimax = train_df[exog_features].values

y_test_sarimax = test_df[target].values
X_test_sarimax = test_df[exog_features].values

# Define SARIMAX order and seasonal order
order = (1, 1, 1)
seasonal_order = (1, 1, 1, 24)

# Create and fit SARIMAX model
print("\nFitting SARIMAX model...")

# Initialize the model
model = SARIMAX(
    y_train_sarimax,
    exog=X_train_sarimax,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

# Fit the model
try:
    results = model.fit(disp=False, maxiter=100)
    model_fit_success = True
except Exception as e:
    print(f"Error fitting model: {e}")
    print("Trying simplified model...")
    
    # Try a simplified model with no seasonality
    try:
        order = (1, 0, 1)
        seasonal_order = (0, 0, 0, 0)
        model = SARIMAX(
            y_train_sarimax,
            exog=X_train_sarimax,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False, maxiter=100)
        model_fit_success = True
    except Exception as e:
        print(f"Error fitting simplified model: {e}")
        model_fit_success = False

# Evaluate the model if it was successfully fit
if model_fit_success:
    # Make predictions
    print("\nMaking predictions...")
    
    # For training data
    y_train_pred = results.predict(exog=X_train_sarimax)
    
    # For test data - using out-of-sample forecast
    # Need to specify the start and end to ensure correct length
    forecast_start = len(y_train_sarimax)
    forecast_end = forecast_start + len(y_test_sarimax) - 1
    y_test_pred = results.predict(start=forecast_start, end=forecast_end, exog=X_test_sarimax)
    
    # Double-check prediction length matches test data length
    if len(y_test_pred) != len(y_test_sarimax):
        print(f"Warning: Length mismatch! Test data: {len(y_test_sarimax)}, Predictions: {len(y_test_pred)}")
        if len(y_test_pred) > len(y_test_sarimax):
            y_test_pred = y_test_pred[:len(y_test_sarimax)]
        else:
            # Pad with NaN or previous values if predictions are shorter
            padding = np.full(len(y_test_sarimax) - len(y_test_pred), np.nan)
            y_test_pred = np.append(y_test_pred, padding)
            # Fill NaN values with the last valid prediction
            for i in range(len(y_test_pred)):
                if np.isnan(y_test_pred[i]) and i > 0:
                    y_test_pred[i] = y_test_pred[i-1]

    
    # Function to calculate metrics
    def calculate_metrics(y_true, y_pred, set_name):
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        # Calculate R² (may be negative if predictions are worse than mean)
        ss_total = np.sum((y_true - np.mean(y_true))**2)
        ss_residual = np.sum((y_true - y_pred)**2)
        r2 = 1 - (ss_residual / ss_total) if ss_total > 0 else 0
        
        # Calculate MAPE (Mean Absolute Percentage Error)
        # Avoid division by zero by adding a small epsilon where y_true is zero
        epsilon = 1e-10
        abs_percentage_errors = np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon)) * 100
        mape = np.mean(abs_percentage_errors)
        
        print(f"{set_name} Set Metrics:")
        print(f"Mean Absolute Error (MAE): {mae:.2f}")
        print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
        print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
        print(f"R² Score: {r2:.4f}")
        print("-" * 30)
        
        return mae, rmse, r2, mape
    
    # Calculate metrics for training and test sets
    print("\nModel Evaluation:")
    print("-" * 50)
    train_mae, train_rmse, train_r2, train_mape = calculate_metrics(y_train_sarimax, y_train_pred, "Training")
    test_mae, test_rmse, test_r2, test_mape = calculate_metrics(y_test_sarimax, y_test_pred, "Test")

In [ ]:
os.makedirs('models', exist_ok=True)
model_filename = "models/sarimax_power_consumption_app1.pkl"
#model_filename = "models/sarimax_power_consumption_app2.pkl"

with open(model_filename, 'wb') as f:
    pickle.dump(results, f)